In [1]:
import cv2
import mediapipe as mp
import numpy as np
import json
import os
import math
import pyttsx3
from datetime import datetime
import threading
import time
import queue

2026-04-16 13:52:56.448343: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 13:52:56.451317: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-16 13:52:56.507668: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-16 13:52:56.512290: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 13:52:57.645980: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

In [2]:
class MudraVaani:
    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7
        )
        self.mp_draw = mp.solutions.drawing_utils
        self.tts_queue = queue.Queue()

        threading.Thread(target=self.tts_worker, daemon=True).start()
        
        self.mudras = ['pathaka', 'tripathaka', 'ardhapathaka', 'kartarimukha', 'mayura', 'katakhamukha', 'hamsasya']
        self.mudra_keys = {
            '1': 'pathaka',
            '2': 'tripathaka', 
            '3': 'ardhapathaka',
            '4': 'kartarimukha',
            '5': 'mayura',
            'k': 'katakhamukha',
            'p': 'hamsasya'
        }
        self.mudra_descriptions = {
            'pathaka': 'Pathaka - Keep your palm open and upright, and the fingers together. Bend the thumb slightly to touch the palm.',
            'tripathaka': 'Tripathaka - From Pathaka, bend the ring finger. Keep the other fingers extended, thumb must touch the palm.',
            'ardhapathaka': 'Ardhapathaka - From Tripathaka, bend the little finger too. Keep the ring finger bent, and the thumb must touch the palm. Fingers must be kept close.',            
            'kartarimukha': 'kartarimukha - Make a scissor symbol, by joining the thumb to the tips of the ring and the little finger. Seperate the index and the middle finger slightly.',
            'mayura': 'mayura - From Tripathaka, touch the thumb to the tip of ring finger, keep the other fingers together and extended.',
            'katakhamukha': 'Katakhamukha - Keep all the fingers straight, and spread them apart. Join the index, middle and the thumb fingers together. ',
            'hamsasya': 'Hamsapksham - Keep all the fingers straight, and spread them apart. Join the index and the thumb fingers together. '
        }
        
        self.reference_data = {}
        if os.path.exists('mudra_references.json'):
            with open('mudra_references.json', 'r') as f:
                self.reference_data = json.load(f)
            self.speak("Loaded saved mudra references")
        else:
            self.speak("No saved references found. Please use teacher mode to create them.")
        
        
        self.current_mudra_index = 0
        self.last_feedback_time = time.time()
        self.feedback_interval = 3   
        self.speaking = False
        self.mudra_locked = False
        
                
    
    def save_references(self):
        with open('mudra_references.json', 'w') as f:
            json.dump(self.reference_data, f, indent=2)
        self.speak("Mudra references saved successfully")

    
    def tts_worker(self):
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 1.0)
    
        while True:
            text = self.tts_queue.get()
            if text is None:
                break
    
            self.speaking = True
            engine.say(text)
            engine.runAndWait()
            self.speaking = False
                
    def speak(self, text, wait=False):
        print(text)
    
        while self.tts_queue.qsize() > 2:
            try:
                self.tts_queue.get_nowait()
            except queue.Empty:
                break
    
        self.tts_queue.put(text)
    
        if wait:
            while self.speaking:
                time.sleep(0.01)
    
    def calculate_angle(self, p1, p2, p3):
        v1 = np.array(p1) - np.array(p2)
        v2 = np.array(p3) - np.array(p2)
        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
    
    def calculate_distance(self, p1, p2):
        return np.linalg.norm(np.array(p1) - np.array(p2))
    
    def extract_hand_features(self, hand_landmarks):
        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.append([lm.x, lm.y, lm.z])
        scale_ref = self.calculate_distance(landmarks[0], landmarks[9]) + 1e-6
        
        features = {
            'landmarks': landmarks,
            'finger_angles': {},
            'finger_distances': {},
            'finger_straightness': {}
        }
        
        fingers = {
            'thumb': [1, 2, 3, 4],
            'index': [5, 6, 7, 8],
            'middle': [9, 10, 11, 12],
            'ring': [13, 14, 15, 16],
            'little': [17, 18, 19, 20]
        }
        
        for key, val in fingers.items():
            angles = []
            for i in range(2):
                p1 = landmarks[val[i]]
                p2 = landmarks[val[i+1]]
                p3 = landmarks[val[i+2]]
                angle = self.calculate_angle(p1, p2, p3)
                angles.append(angle)
            features['finger_angles'][key] = angles
            
            tip = landmarks[val[3]]
            base = landmarks[val[0]]
            distance = self.calculate_distance(tip, base) / scale_ref #required??
            features['finger_distances'][key] = distance
            
            avg_angle = np.mean(angles) if angles else 0
            features['finger_straightness'][key] = avg_angle
        
        features['thumb_to_palm'] = self.calculate_distance(landmarks[4], landmarks[5]) / scale_ref
        features['fingers_spread'] = self.calculate_distance(landmarks[6], landmarks[18]) / scale_ref #calculates only from index??
        features['thumb_to_index'] = self.calculate_distance(landmarks[4], landmarks[8]) / scale_ref
        features['thumb_to_middle'] = self.calculate_distance(landmarks[4], landmarks[12]) / scale_ref
        features['thumb_to_ring'] = self.calculate_distance(landmarks[4], landmarks[16]) / scale_ref
        features['thumb_to_little'] = self.calculate_distance(landmarks[4], landmarks[20]) / scale_ref        
        return features

    def get_finger_state(self, angle):
        if angle>160:
            return "extended"
        elif angle>100:
            return "half"
        else:
            return "curled"

    def extract_symbolic_features(self, features):
        landmarks = features['landmarks']
    
        states = {}
        for finger in ['thumb', 'index', 'middle', 'ring', 'little']:
            angle = features['finger_straightness'][finger]
            states[finger] = self.get_finger_state(angle)

        relations = {}
        if features['thumb_to_palm']<0.31:
            relations["thumb_palm"] = "touching"
        else:
            relations["thumb_palm"] = "away"
        for finger in ['index', 'middle', 'ring', 'little']:
            if features[f'thumb_to_{finger}']<0.32:
                relations[f"thumb_{finger}"] = "touching"
            else:
                relations[f"thumb_{finger}"] = "away"
        if features['fingers_spread']<0.6:
            relations["fingers"] = "close"
        else:
            relations["fingers"] = "spread"    
        
        return {
            "states": states,
            "relations": relations
        }
    
    def compare_symbolic(self, student, reference, mudra_name):
        feedback = []
        relation_error = 0
        ignore_thumb_palm = False
        for rel in reference["relations"]:
            if rel!="thumb_palm" and reference["relations"][rel] == "touching":
                ignore_thumb_palm = True
                break
    
        for rel in reference["relations"]:
            if student["relations"][rel] != reference["relations"][rel]:
                relation_error += 1
                if rel == "fingers":
                    if reference["relations"][rel] == "close":
                        feedback.append("Bring your fingers closer together")
                    else:
                        feedback.append("Spread your fingers apart")   
                elif rel == "thumb_palm":
                    if reference["relations"][rel] == "touching":
                        feedback.append("Touch your thumb to your palm")
                    elif not ignore_thumb_palm:
                        feedback.append("Keep your thumb away from the palm") 
                else:
                    for finger in ['index', 'middle', 'ring', 'little']:
                        if rel == f"thumb_{finger}":
                            if reference["relations"][rel] == "touching":
                                feedback.append(f"Bring your thumb and {finger} finger together")
                            else:
                                feedback.append(f"Keep your thumb and {finger} finger away")

        if relation_error <= 1:
            for finger in ['thumb', 'index', 'middle', 'ring', 'little']:
                if student["states"][finger] != reference["states"][finger]:
                    if reference["states"][finger] == "extended":
                        feedback.append(f"Straighten your {finger} finger")
                    elif reference["states"][finger] == "half":
                        feedback.append(f"Slightly curl your {finger} finger")
                    elif reference["states"][finger] == "curled":
                        feedback.append(f"Curl your {finger} finger")
                    else:
                        feedback.append(f"Adjust your {finger} finger slightly")
    
    
        return feedback
    
    def teacher_mode(self):
        self.speak("Welcome to teacher mode", wait=True)
        self.speak("Press keys 1 through 5 to select mudras. Press SPACE to capture. Press Q to quit", wait=True)
        self.speak("Press H at any time to hear help", wait=True)
        
        cap = cv2.VideoCapture(0)
        current_mudra = None
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame = cv2.flip(frame, 1)
            h, w, c = frame.shape
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.hands.process(rgb_frame)
            
            cv2.putText(frame, "TEACHER MODE", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            
            if current_mudra:
                cv2.putText(frame, f"Current: {current_mudra}", (10, 70),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

                cv2.putText(frame, "Press space to capture", (10, 130),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                
                if current_mudra in self.reference_data:
                    cv2.putText(frame, "[Already saved]", (10, 160),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            
            y_pos = h - 150
            cv2.putText(frame, "KEYS: 1=Pathaka 2=Tripathaka 3=Ardhapathaka", 
                       (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, "       5=mayura k=katakhamukha p=hamsasya", 
                       (10, y_pos + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, "      SPACE=Capture H=Help Q=Quit", 
                       (10, y_pos + 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    self.mp_draw.draw_landmarks(
                        frame, hand_landmarks, self.mp_hands.HAND_CONNECTIONS)
                    
                    if current_mudra:
                        cv2.putText(frame, "Ready to capture", (10, 190),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            cv2.imshow('Teacher Mode', frame)
            
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q'):
                self.speak("Exiting teacher mode", wait=True)
                break
            elif key == ord('h'):
                self.speak("Teacher mode", wait=True)
            elif chr(key) in self.mudra_keys:
                current_mudra = self.mudra_keys[chr(key)]
                self.speak(f"Selected {current_mudra}. {self.mudra_descriptions[current_mudra]}", wait=True)
            elif key == ord(' '):
                if current_mudra:
                    if results.multi_hand_landmarks:
                        features = self.extract_hand_features(results.multi_hand_landmarks[0])
                        symbolic = self.extract_symbolic_features(features)
                        
                        self.reference_data[current_mudra] = symbolic
                        self.save_references()
                        self.speak(f"{current_mudra} mudra captured successfully!", wait=True)
                    else:
                        self.speak("No hand detected. Please show your hand clearly", wait=True)
                else:
                    self.speak("Please select a mudra first by pressing keys 1 through 5", wait=True)
        
        cap.release()
        cv2.destroyAllWindows()
    
    def student_mode(self):
        """Student mode: Practice mudras with keyboard selection and voice feedback"""
        self.speak("Welcome to student mode", wait=True)
        
        if not self.reference_data:
            self.speak("No reference mudras found. Please ask teacher to create references first.", wait=True)
            return
        
        available_mudras = list(self.reference_data.keys())
        self.speak(f"Available mudras: {', '.join(available_mudras)}", wait=True)
        self.speak("Press keys 1 through 5 to select mudra, or Q to quit", wait=True)
        self.speak("Press H at any time to hear help", wait=True)
        
        cap = cv2.VideoCapture(0)
        current_mudra = None
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame = cv2.flip(frame, 1)
            h, w, c = frame.shape
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.hands.process(rgb_frame)
            
            cv2.putText(frame, "STUDENT MODE", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            
            if current_mudra:
                cv2.putText(frame, f"Practicing: {current_mudra}", (10, 70),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            else:
                cv2.putText(frame, "Select a mudra (press 1-5)", (10, 70),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
            
            y_pos = h - 150
            cv2.putText(frame, "KEYS: 1=Pathaka 2=Tripathaka 3=Ardhapathaka", 
                       (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, "      5=mayura k=katakhamukha p=hamsasya", 
                       (10, y_pos + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, "      H=Help Q=Quit", 
                       (10, y_pos + 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            if results.multi_hand_landmarks and current_mudra:
                for hand_landmarks in results.multi_hand_landmarks:
                    self.mp_draw.draw_landmarks(
                        frame, hand_landmarks, self.mp_hands.HAND_CONNECTIONS)
                    
                    features = self.extract_hand_features(hand_landmarks)

                    student_symbolic = self.extract_symbolic_features(features)
                    reference_symbolic = self.reference_data[current_mudra]
                    
                    feedback = self.compare_symbolic(student_symbolic, reference_symbolic, current_mudra)
                    
                    current_time = time.time()
                    if not self.mudra_locked:
                        if current_time - self.last_feedback_time > self.feedback_interval and not self.speaking:
                            if feedback:
                                feedback_text = ". ".join(feedback[:2])
                                self.speak(feedback_text)
                            else:
                                self.speak("Excellent! Your mudra is correct!")
                                self.mudra_locked = True
                            self.last_feedback_time = current_time
                    
                    y_offset = 100
                    if not self.mudra_locked:
                        for fb in feedback[:3]:
                            cv2.putText(frame, fb, (10, y_offset),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                            y_offset += 25
                    
                    if self.mudra_locked:
                        cv2.putText(frame, "Perfect! Well done!", (10, 100),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            cv2.imshow('Student Mode', frame)
            
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q'):
                self.speak("Exiting student mode", wait=True)
                break
            elif key == ord('h'):
                help_text = "Student mode. Press 1 for Pathaka, 2 for Tripathaka, 3 for Ardhapathaka, 5 for mayura, k for katakhamukha, p for hamsasya. Show your hand to get feedback. Press Q to quit."
                self.speak(help_text, wait=True)
            elif chr(key) in self.mudra_keys:
                mudra_name = self.mudra_keys[chr(key)]
                if mudra_name in self.reference_data:
                    current_mudra = mudra_name
                    self.mudra_locked = False
                    self.show_perfect = False
                    self.speak(f"Starting practice for {current_mudra}. {self.mudra_descriptions[current_mudra]}", wait=True)
                    self.speak("Show your hand when ready", wait=True)
                    self.last_feedback_time = time.time()
                else:
                    self.speak(f"{mudra_name} has not been saved yet. Please ask teacher to capture it first.", wait=True)
        
        cap.release()
        cv2.destroyAllWindows()
    
    def run(self):
        self.speak("Welcome to MudraVaani", wait=True)
        self.speak("All controls are via keyboard. Voice output will guide you.", wait=True)
        
        cv2.namedWindow('Mudra Teacher - Main Menu')
        blank_img = np.zeros((200, 600, 3), dtype=np.uint8)
        
        while True:
            menu_img = blank_img.copy()
            cv2.putText(menu_img, "MAIN MENU", (200, 40),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            cv2.putText(menu_img, "T = Teacher Mode  |  S = Student Mode", (80, 80),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            cv2.putText(menu_img, "H = Help  |  Q = Quit", (150, 110),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            cv2.putText(menu_img, "Press a key", (220, 160),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            
            cv2.imshow('Mudra Teacher - Main Menu', menu_img)
            
            self.speak("Press T for teacher mode, S for student mode, H for help, or Q to quit", wait=False)
            
            key = cv2.waitKey(0) & 0xFF
            
            if key == ord('t'):
                cv2.destroyWindow('Mudra Teacher - Main Menu')
                self.teacher_mode()
                cv2.namedWindow('Mudra Teacher - Main Menu')
            elif key == ord('s'):
                cv2.destroyWindow('Mudra Teacher - Main Menu')
                self.student_mode()
                cv2.namedWindow('Mudra Teacher - Main Menu')
            elif key == ord('h'):
                help_text = "Welcome to MudraVaani. In teacher mode, press keys 1 through 5 to select mudras and SPACE to capture them. In student mode, press keys 1 through 5 to practice mudras and receive voice feedback. All five mudras are: 1 is Pathaka, 2 is Tripathaka, 3 is Ardhapathaka, 5 is mayura."
                self.speak(help_text, wait=True)
            elif key == ord('q'):
                self.speak("Thank you for using MudraVaani. Goodbye!", wait=True)
                break
            else:
                self.speak("Invalid key. Please press T, S, H, or Q", wait=True)
            

        
        cv2.destroyAllWindows()



In [3]:
app = MudraVaani()
app.run()

I0000 00:00:1776327782.444575    6142 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776327782.452124    6759 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: Mesa Intel(R) Arc(tm) Graphics (MTL)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
QFontDatabase: Cannot find font directory /home/ananya-shetty/.pyenv/versions/3.8.18/envs/py38_env/lib/python3.8/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ananya-shetty/.pyenv/versions/3.8.18/envs/py38_env/lib/python3.8/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/ananya-shetty/.pyenv/versions/3.8.18/envs/py38_env/lib/python3.8/site-packages/cv2/qt/fonts

Loaded saved mudra references
Welcome to MudraVaani
All controls are via keyboard. Voice output will guide you.
Press T for teacher mode, S for student mode, H for help, or Q to quit
Welcome to teacher mode
Press keys 1 through 5 to select mudras. Press SPACE to capture. Press Q to quit
Press H at any time to hear help
Selected pathaka. Pathaka - Keep your palm open and upright, and the fingers together. Bend the thumb slightly to touch the palm.
Mudra references saved successfully
pathaka mudra captured successfully!
Exiting teacher mode
Press T for teacher mode, S for student mode, H for help, or Q to quit
Welcome to student mode
Available mudras: pathaka, tripathaka, ardhapathaka, kartarimukha, mayura, katakhamukha, hamsasya
Press keys 1 through 5 to select mudra, or Q to quit
Press H at any time to hear help
Starting practice for pathaka. Pathaka - Keep your palm open and upright, and the fingers together. Bend the thumb slightly to touch the palm.
Show your hand when ready
Bring y

# 